# Model Evaluation

Comprehensive evaluation of the trained ANN model on the test dataset.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy.sparse import load_npz

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor
from src.ann_model import DocumentClassifierANN

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)

Path('../results/figures').mkdir(parents=True, exist_ok=True)
Path('../results/metrics').mkdir(parents=True, exist_ok=True)
Path('../results/reports').mkdir(parents=True, exist_ok=True)

## 1. Load Test Data and Model

In [ ]:
# Load test data
X_test = load_npz('../data/processed/X_test.npz')
y_test = np.load('../data/processed/y_test.npy')

# Load preprocessing artifacts
preprocessor = TextPreprocessor('../config/config.yaml')
preprocessor.load_artifacts()

# Load model
ann = DocumentClassifierANN('../config/config.yaml')
ann.load_model('../models/document_classifier.keras')

# Load category names
loader = DataLoader('../config/config.yaml')
dataset = loader.load_dataset()
target_names = dataset['target_names']

print(f"Test data: {X_test.shape}")
print(f"Test labels: {y_test.shape}")
print(f"Categories: {len(target_names)}")

## 2. Predict on Test Set

In [ ]:
# Convert to dense for prediction
X_test_dense = X_test.toarray()

# Predict
y_probs = ann.predict(X_test_dense)
y_pred = np.argmax(y_probs, axis=1)

# Top-5 predictions
top5_indices = np.argsort(y_probs, axis=1)[:, -5:][:, ::-1]
top5_correct = np.any(top5_indices == y_test.reshape(-1, 1), axis=1)
top5_acc = top5_correct.mean()

print(f"Test samples: {len(y_test)}")
print(f"Predictions shape: {y_pred.shape}")
print(f"Probabilities shape: {y_probs.shape}")

## 3. Overall Metrics

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("=== OVERALL METRICS ===")
print(f"Accuracy:           {accuracy:.4f}")
print(f"Top-5 Accuracy:     {top5_acc:.4f}")
print(f"Precision (Macro):  {precision_macro:.4f}")
print(f"Precision (Weighted): {precision_weighted:.4f}")
print(f"Recall (Macro):     {recall_macro:.4f}")
print(f"Recall (Weighted):  {recall_weighted:.4f}")
print(f"F1-Score (Macro):   {f1_macro:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")

## 4. Classification Report

In [ ]:
# Detailed classification report
report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True, zero_division=0)
report_text = classification_report(y_test, y_pred, target_names=target_names, zero_division=0)

print(report_text)

In [ ]:
# Save classification report
report_df = pd.DataFrame(report).T
report_df.to_csv('../results/reports/classification_report.csv', float_format='%.4f')

with open('../results/reports/classification_report.txt', 'w') as f:
    f.write("CLASSIFICATION REPORT\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Overall Accuracy: {accuracy:.4f}\n")
    f.write(f"Top-5 Accuracy: {top5_acc:.4f}\n")
    f.write(f"Macro Precision: {precision_macro:.4f}\n")
    f.write(f"Weighted Precision: {precision_weighted:.4f}\n")
    f.write(f"Macro Recall: {recall_macro:.4f}\n")
    f.write(f"Weighted Recall: {recall_weighted:.4f}\n")
    f.write(f"Macro F1-Score: {f1_macro:.4f}\n")
    f.write(f"Weighted F1-Score: {f1_weighted:.4f}\n\n")
    f.write(report_text)

print("Classification report saved to ../results/reports/")

## 5. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=9)
axes[0].tick_params(axis='y', rotation=0, labelsize=9)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[1], cbar_kws={'label': 'Proportion'})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].tick_params(axis='x', rotation=45, labelsize=9)
axes[1].tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig('../results/figures/confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Save confusion matrix
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
cm_df.to_csv('../results/reports/confusion_matrix.csv')

cm_norm_df = pd.DataFrame(cm_normalized, index=target_names, columns=target_names)
cm_norm_df.to_csv('../results/reports/confusion_matrix_normalized.csv', float_format='%.4f')

print("Confusion matrices saved to ../results/reports/")

## 6. Per-Category Performance Visualization

In [ ]:
# Extract per-class metrics
categories = target_names
precision = [report[cat]['precision'] for cat in categories]
recall = [report[cat]['recall'] for cat in categories]
f1 = [report[cat]['f1-score'] for cat in categories]
support = [report[cat]['support'] for cat in categories]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Precision
bars1 = axes[0, 0].barh(range(len(categories)), precision, color='#2E86C1', alpha=0.8)
axes[0, 0].set_yticks(range(len(categories)))
axes[0, 0].set_yticklabels(categories, fontsize=9)
axes[0, 0].set_xlabel('Precision')
axes[0, 0].set_title('Per-Class Precision', fontsize=13, fontweight='bold')
axes[0, 0].set_xlim([0, 1.05])
axes[0, 0].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars1, precision):
    axes[0, 0].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

# Recall
bars2 = axes[0, 1].barh(range(len(categories)), recall, color='#27AE60', alpha=0.8)
axes[0, 1].set_yticks(range(len(categories)))
axes[0, 1].set_yticklabels(categories, fontsize=9)
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_title('Per-Class Recall', fontsize=13, fontweight='bold')
axes[0, 1].set_xlim([0, 1.05])
axes[0, 1].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars2, recall):
    axes[0, 1].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

# F1-Score
bars3 = axes[1, 0].barh(range(len(categories)), f1, color='#E74C3C', alpha=0.8)
axes[1, 0].set_yticks(range(len(categories)))
axes[1, 0].set_yticklabels(categories, fontsize=9)
axes[1, 0].set_xlabel('F1-Score')
axes[1, 0].set_title('Per-Class F1-Score', fontsize=13, fontweight='bold')
axes[1, 0].set_xlim([0, 1.05])
axes[1, 0].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars3, f1):
    axes[1, 0].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

# Support
bars4 = axes[1, 1].barh(range(len(categories)), support, color='#8E44AD', alpha=0.8)
axes[1, 1].set_yticks(range(len(categories)))
axes[1, 1].set_yticklabels(categories, fontsize=9)
axes[1, 1].set_xlabel('Support (Test Samples)')
axes[1, 1].set_title('Per-Class Support', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars4, support):
    axes[1, 1].text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/per_class_metrics.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 7. Training History

In [ ]:
# Load training history
with open('../results/metrics/training_history.json', 'r') as f:
    history = json.load(f)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['accuracy']) + 1)

axes[0].plot(epochs, history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
axes[0].plot(epochs, history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
axes[0].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

axes[1].plot(epochs, history['loss'], 'b-', label='Training Loss', linewidth=2)
axes[1].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
axes[1].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/training_history.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Error Analysis

In [ ]:
# Find misclassified samples
misclassified = y_test != y_pred
misclassified_indices = np.where(misclassified)[0]

print(f"Total misclassified: {len(misclassified_indices)} out of {len(y_test)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)")

# Most confused category pairs
confusion_pairs = []
for i in range(len(target_names)):
    for j in range(len(target_names)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'true': target_names[i],
                'predicted': target_names[j],
                'count': cm[i, j]
            })

confusion_df = pd.DataFrame(confusion_pairs).sort_values('count', ascending=False)
print("\n=== TOP 10 MOST CONFUSED PAIRS ===")
print(confusion_df.head(10).to_string(index=False))

In [ ]:
# Save all evaluation metrics
metrics = {
    'accuracy': float(accuracy),
    'top5_accuracy': float(top5_acc),
    'precision_macro': float(precision_macro),
    'precision_weighted': float(precision_weighted),
    'recall_macro': float(recall_macro),
    'recall_weighted': float(recall_weighted),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'per_class': report,
    'confusion_matrix': cm.tolist()
}

with open('../results/metrics/evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("All evaluation metrics saved to ../results/metrics/evaluation_metrics.json")
print("\n=== EVALUATION COMPLETE ===")

## 9. Summary

### Evaluation Summary

The trained MLP model has been evaluated on the held-out test set with the following key metrics:

- **Accuracy**: Overall classification accuracy
- **Top-5 Accuracy**: Whether true label is in top 5 predictions
- **Macro F1**: Unweighted mean of per-class F1 scores
- **Weighted F1**: Support-weighted mean of per-class F1 scores

### Generated Artifacts:

- `results/metrics/evaluation_metrics.json` - All metrics in JSON
- `results/reports/classification_report.csv/txt` - Detailed classification report
- `results/reports/confusion_matrix.csv` - Raw confusion matrix
- `results/reports/confusion_matrix_normalized.csv` - Normalized confusion matrix
- `results/figures/confusion_matrix.png` - Confusion matrix visualization
- `results/figures/per_class_metrics.png` - Per-class precision/recall/F1
- `results/figures/training_history.png` - Training curves

### Next Steps:

1. Review per-class performance for potential improvements
2. Analyze confusion patterns for feature engineering ideas
3. Launch Streamlit app for interactive testing: `streamlit run app/Home.py`